Slowly Changing Dimension(SCD Type-1)
merge(upsert) operation
-we need to update the target table based on the incoming source data

-we need to compare the data of target and source and decide whether to insert
                                                                     or update

-while comparing target and incoming source data

 if there is a match found then
     -the target data is updated based on source data
 if there is no match in target table
     -that means we recieved new data
     -means new rec will be inserted into target table
 This operation is called as upsert (or) merge operation


when datalake was introduced we were unable to perform this merge(upsert)
operation

when databricks introduced deltalake,we are able to perform this
merge(upsert) operation.

merge operation--->2ways

1)sparksql

2)pyspark

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

schema=StructType([
                 StructField("eid",IntegerType(),True),
                 StructField("ename",StringType(),True),
                 StructField("salary",IntegerType(),True),
                 StructField("dept",StringType(),True),
                 StructField("city",StringType(),True)
])


create dataframe

In [0]:
data=[(1,"Raj",50000,"IT","Bangalore")]

df=spark.createDataFrame(data=data,schema=schema)
display(df) 

This(above) is our source data---->df

Now i will create a deltatable


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.demo_db.dim_employee (eid INT, ename STRING, salary INT, dept STRING, city STRING)
USING DELTA
LOCATION "abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/dim_employee1"

In [0]:
%sql
select * from dev.demo_db.dim_employee; 

This delta table(dim_employee)-----> is our target
              
              
source  -------------> a df

Method1 : sparksql

In spark sql ---->to perform merge operation 

both source and target should be equal(same)
 i.e both should be tables


our source is df---->convert that df to tempview or temptable

In [0]:
df.show()
df.createOrReplaceTempView("source_view")

see the source_view

In [0]:
%sql
select * from source_view;  

In [0]:
%sql
select * from dev.demo_db.dim_employee;


delta table(dim_employee)-----> is our target

source -------------> a df

terget is empty so we perform the merge operation

In [0]:
%sql
MERGE INTO dev.demo_db.dim_employee AS target
USING source_view AS source
   ON target.eid=source.eid
   WHEN MATCHED THEN 
   UPDATE  
     SET target.ename = source.ename,
     target.salary = source.salary,
     target.dept = source.dept,
     target.city = source.city
   WHEN NOT MATCHED THEN
   INSERT (eid, ename, salary, dept, city) values (source.eid, source.ename, source.salary, source.dept, source.city);


One record inserted.
Now see the output

In [0]:
%sql
select * from dev.demo_db.dim_employee;

Great - 1 record inserted

process: merge into target

 using source
 when matched
   -->update
 Else
   --->insert

here target was empty--->so no match
                      -->so we insert


for update

I will create some sample data


In [0]:
data=[(1,"Raj",50000,"Sales","Hyderabad"),(2,"Suresh",60000,"IT","Bangalore")]

df=spark.createDataFrame(data=data,schema=schema)
display(df)

In [0]:
df.createOrReplaceTempView("source_view")

In [0]:
%sql
select * from source_view

In [0]:
%sql
select * from dev.demo_db.dim_employee;

observe data in dim_employee and df

dim_employee has 1 record.

df has 2 records, source_view is temp view name.

so when we perform merge: 

Here one record match-------->update happens

and one new record not match-->insert happens

based on eid--->match occured--->So it updates city with latest value i.e to Hyderabad

also should insert a new record

Now run the merge operation again


In [0]:
%sql
MERGE INTO dev.demo_db.dim_employee AS target
USING source_view AS source
   ON target.eid=source.eid
   WHEN MATCHED THEN 
   UPDATE  
     SET target.ename = source.ename,
     target.salary = source.salary,
     target.dept = source.dept,
     target.city = source.city
   WHEN NOT MATCHED THEN
   INSERT (eid, ename, salary, dept, city) values (source.eid, source.ename, source.salary, source.dept, source.city);

Now see the output of dim_employee table, it should have 2 records now.

In [0]:
%sql
select * from dev.demo_db.dim_employee;

# method 2-->pyspark style


In [0]:
data=[(2,"Suresh",70000,"IT","Bangalore"),(3,"Anil",60000,"IT","Bangalore")]

df=spark.createDataFrame(data=data,schema=schema)
display(df)


Here one record match--->update

and one new record-->insert

here in pyspark,create dataframe is enough,no need to create temp view

In [0]:
# Now creating instance of deltatable

from delta.tables import DeltaTable
delta_df=DeltaTable.forPath(spark,"abfss://rawdata@mystoragelakeadb6pmgroup.dfs.core.windows.net/external1/dim_employee1")

In [0]:
delta_df.alias("target").merge(
    source=df.alias("source"),
    condition="target.eid=source.eid"
).whenMatchedUpdate(set=
                    {
                        "ename": "source.ename",
                        "salary": "source.salary",
                        "dept": "source.dept",
                        "city": "source.city"

                    }
             ).whenNotMatchedInsert(values=
               {
                        "eid": "source.eid",
                        "ename": "source.ename",
                        "salary": "source.salary",
                        "dept": "source.dept",
                        "city": "source.city"
                    
               }

).execute()

In [0]:
delta_df.toDF().show()

GREAT: 2 records Updated.
And 1 record Inserted. 

In [0]:
%sql
select * from dev.demo_db.dim_employee;

See CELL no 6. 

Table dev.demo_db.Dim_employee located at rawdata container(external1->dim_employee1).